In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path
import optuna
from optuna.trial import Trial
import joblib
import random
from functools import lru_cache

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.esic_v1 import *
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *
from sj_utils.collection_utils import SafetyDict

In [ ]:
from rt_whisper import streamers
from rt_whisper.data import Param, Result

In [ ]:
SOURCE = "/workspaces/dev/datasets/ESIC-v1.1/v1.1/dev"
STUDY = "/workspaces/dev/study/esic/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [ ]:
src = Path(SOURCE)
study_path = Path(STUDY) / "study_non_filter.pkl"
study_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
src_folder = search_all_data(src)

In [ ]:
@lru_cache(maxsize=128)
def load_mp4(mp4, sr=SAMPLE_RATE):
    return load_audio_from_mp4(mp4, sr=sr)

In [ ]:
def normalize_text(text):
    return normalize_text_only_en(text).upper()

In [ ]:
def objective(trial:Trial):
    hyperparameters = SafetyDict({
        "whisper": {
            "model_options": {
                "model_size_or_path": "large-v3",
                "device": "cuda",
                "compute_type": "float16",
            },
            "transcribe_options": {
                "beam_size":5,
                "vad_filter": False,
                "temperature": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
            }
        },
        "silero_vad": {
            "model_options": {},
            "run_options": {}
        },
        "asr": {
            "max_overlap_duration": trial.suggest_int(
                "overlap_duration", 8000, 112000, step=8000
            )
        },
        "position_weighted_filter": {
            "boundary":trial.suggest_int(
                "boundary", 4000, 16000, step=100
            )
        },
        "duration_filter":{
            "z_thresh":{
                "default": 2.0,
                "en": trial.suggest_float("df_z_thresh", 3.0, 5.0, step = 0.1)
            },
            "min_duration": {
                "default": 160,
                "en": trial.suggest_float("df_min_duration", 0, 16000, step = 160)
            },
        },
        "probability_filter":{
            "z_thresh":{
                "default": 3.0,
                "en": trial.suggest_float("pf_z_thresh", 3.0, 5.0, step = 0.1)
            },
            "min_prob": {
                "default": 1.0,
                "en": trial.suggest_float("pf_min_prob", 0, 1, step = 0.05)
            },
        },
        "selector":{
            "search_range_sc": {
                "default": 24000,
                "en": trial.suggest_int("search_range_sc", 0, 96000, step=1000)
            },
            "threshold":{
                "default": 0.5,
                "en": trial.suggest_float("threshold", 0, 1, step=0.05)
            },
            "padding": {
                "default": 3200,
                "en": trial.suggest_int("padding", 0, 32000, step=100)
            },
            "tolerance": {
                "default": 8000,
                "en": trial.suggest_int("tolerance", 0, 32000, step=100)
            }
        },
    })

    token_streamer = streamers.get_token_streamer_with_vad_v2(hyperparameter=hyperparameters)

    def transcriber(mp4:Path) -> TRNFormat:
        audio, _ = load_mp4(mp4, sr=SAMPLE_RATE)

        completed = []
        param = Param()
        for segment in segment_audio(audio):
            param.chunk = segment
            param.language="en"
            result:Result = token_streamer.process(param)
            completed.extend(result.completed)
            param.update(result, update_prompt=True)
        completed.extend(result.candidate)

        text = " ".join([s.text for s in completed])
        text = normalize_text(text)

        return text

    samples = random.sample(src_folder, 5)

    data = {}
    for sample in samples:
        trans_txt = search_file_from_dir(sample, "o")
        ref = trans_txt_to_sclite_trn(trans_txt, normalize_text)
        pred = transcriber(search_file_from_dir(sample, "mp4"))
        hyp = TRNFormat(id = ref.id, text = pred)
        data[f"{sample.parent.stem}_{sample.stem}"] = {"ref": ref,"hyp": hyp}

    concat_result = {}
    for value in data.values():
        for k, v in value.items():
            if k not in concat_result:
                concat_result[k] = []
            concat_result[k].append(v)

    output = sclite_trn(concat_result["ref"], concat_result["hyp"])
    result = parse_sclite_summary(output)

    return result["wer_percent"]

In [ ]:
if study_path.exists():
    study = joblib.load(study_path)
else:
    study = optuna.create_study(direction="minimize")

In [ ]:
for _ in range(500):
    study.optimize(objective, n_trials=5)
    joblib.dump(study, study_path)

In [ ]:
study.best_value

In [ ]:
study.best_params